In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

# === 1. 读取数据，先确认列名和格式 ===
df = pd.read_csv('okx_btc_usdt_15m.csv', index_col=0, parse_dates=True)
print(df.columns.tolist())
print(df.head())
# 如果发现index没被正确解析成时间，或者列名是数字/大写，根据实际情况调整上面这行

df = df.sort_index()  # 确保按时间升序，否则后面shift/rolling会算错

# === 2. 计算对数收益率 ===
df['log_ret'] = np.log(df['close'] / df['close'].shift(1))

# === 3. 构造动量信号：过去N根bar的累计对数收益率 ===
N = 288  # 约3天（15min×288=72h），可以换成96(1天)/672(1周)对比不同窗口
df['momentum'] = df['log_ret'].rolling(N).sum()

# === 4. 信号转仓位，shift(1)避免用到未来数据 ===
df['signal'] = np.where(df['momentum'] > 0, 1, -1)
df['position'] = df['signal'].shift(1)

# === 5. 策略收益（先看不含交易成本的版本）===
df['strategy_ret'] = df['position'] * df['log_ret']

# === 6. 加一个简单交易成本，换仓才扣费 ===
fee_rate = 0.0005  # 万5，按OKX实际费率可调整
df['trade'] = df['position'].diff().abs()
df['strategy_ret_net'] = df['strategy_ret'] - df['trade'] * fee_rate

# === 7. 单因子回归检验alpha ===
valid = df.dropna(subset=['strategy_ret_net'])
X = sm.add_constant(valid['log_ret'])
model = sm.OLS(valid['strategy_ret_net'], X).fit(cov_type='HAC', cov_kwds={'maxlags': 20})
print(model.summary())

# === 8. 直观看一下净值曲线 ===
cum_strategy = (1 + df['strategy_ret_net']).cumprod()
cum_bh = (1 + df['log_ret']).cumprod()
print(f"策略最终净值: {cum_strategy.iloc[-1]:.2f}")
print(f"买入持有最终净值: {cum_bh.iloc[-1]:.2f}")

['open', 'high', 'low', 'close', 'volume']
                        open     high      low    close      volume
timestamp                                                          
2025-02-25 02:00:00  92032.0  92325.6  91804.0  91822.7   86.625201
2025-02-25 02:15:00  91820.7  92117.9  91600.0  92100.0  118.216880
2025-02-25 02:30:00  92099.9  92329.8  91974.1  92329.7   44.928742
2025-02-25 02:45:00  92329.8  92530.0  92306.7  92384.0   65.882587
2025-02-25 03:00:00  92384.0  92545.9  92160.1  92160.1   51.346619
                            OLS Regression Results                            
Dep. Variable:       strategy_ret_net   R-squared:                       0.027
Model:                            OLS   Adj. R-squared:                  0.027
Method:                 Least Squares   F-statistic:                     21.21
Date:                Thu, 03 Sep 2026   Prob (F-statistic):           4.13e-06
Time:                        11:19:55   Log-Likelihood:             2.3141e+05
No. Obs